# SWE-Inject-Steps — reproduce the main results

This notebook downloads the public **[SWE-Inject-Steps](https://huggingface.co/datasets/ArsalanKaleem/swe-inject-steps)**
dataset and reproduces the paper's main numbers and figures. It runs on CPU, needs no login, and no GPU.

**What it reproduces**
1. The hijack-rate table across model sizes (the capability gradient).
2. Paired significance tests (Holm-adjusted).
3. The URL-free blind spot: a URL-provenance rule catches every URL-bearing attack but no URL-free one.
4. A figure of the capability gradient.


## 1. Download the dataset (no login)

In [ ]:
import json, math, urllib.request, io, re
from collections import Counter

REPO = "ArsalanKaleem/swe-inject-steps"
BASE = f"https://huggingface.co/datasets/{REPO}/resolve/main"
FILES = {"Coder-7B": "coder7b_goalonly.jsonl",
         "Coder-14B": "coder14b_goalonly.jsonl",
         "Coder-32B": "coder32b_goalonly.jsonl"}
URLFREE = {"Coder-14B": "urlfree_coder14b_goalonly.jsonl",
           "Coder-32B": "urlfree_coder32b_goalonly.jsonl"}

def fetch(fname):
    with urllib.request.urlopen(f"{BASE}/{fname}") as r:
        return [json.loads(l) for l in io.TextIOWrapper(r, encoding="utf-8") if l.strip()]

data = {k: fetch(v) for k, v in FILES.items()}
urlfree = {k: fetch(v) for k, v in URLFREE.items()}
print("trajectories per file:", {k: len(v) for k, v in data.items()})

## 2. Helpers

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0.0, 100*(c-h)), min(100.0, 100*(c+h)))

def attacked_tasks(rows):
    return [r for r in rows if r["meta"].get("goal_kind", "none") != "none"]

def rates(rows):
    att = attacked_tasks(rows); n = len(att)
    reached = sum(r["category"] in ("attacked", "failed_attack") for r in att)
    hij = sum(r["category"] == "attacked" for r in att)
    return n, reached, hij

## 3. Main table — hijack rate rises with model capability

In [ ]:
print(f"{'model':11s}{'reached':>14s}{'hijacked/all':>22s}{'hijacked/reached':>18s}")
for lab, rows in data.items():
    n, reached, hij = rates(rows)
    lo, hi = wilson(hij, n)
    rr = f"{reached}/{n} ({100*reached/n:.1f}%)"
    ha = f"{hij}/{n} ({100*hij/n:.1f}% [{lo:.1f},{hi:.1f}])"
    hr = f"{hij}/{reached} ({100*hij/reached:.1f}%)" if reached else "n/a"
    print(f"{lab:11s}{rr:>14s}{ha:>22s}{hr:>18s}")

## 4. Paired significance (exact sign test, Holm-adjusted)

In [ ]:
def sign_p(b, c):
    n = b+c; m = min(b, c)
    return min(1.0, 2*sum(math.comb(n, i) for i in range(m+1))/2**n) if n else 1.0

def paired(a_rows, b_rows):
    A = {r["id"]: r["category"]=="attacked" for r in attacked_tasks(a_rows)}
    B = {r["id"]: r["category"]=="attacked" for r in attacked_tasks(b_rows)}
    sh = set(A) & set(B)
    oa = sum(A[i] and not B[i] for i in sh); ob = sum(B[i] and not A[i] for i in sh)
    return sign_p(oa, ob)

labs = list(data); tests = []
for i in range(len(labs)):
    for j in range(i+1, len(labs)):
        tests.append((f"{labs[i]} vs {labs[j]}", paired(data[labs[i]], data[labs[j]])))
tests.sort(key=lambda t: t[1]); prev = 0; m = len(tests)
for i, (name, p) in enumerate(tests):
    adj = min(1, max(prev, (m-i)*p)); prev = adj
    print(f"{name:24s} raw p={p:.2e}   Holm-adjusted={adj:.2e}")

## 5. The URL-free blind spot\n\nA provenance rule holds any tool call whose arguments contain a value (here an address: URL, domain or email) introduced by tool output. It catches URL-bearing attacks but not attacks whose command names only a local path (`run_local_script`).

In [ ]:
addr = re.compile(r"https?://|[\w.-]+\.(?:com|net|org|io|evil|host)\b|@")
def catchable_by_url_rule(rows):
    out = {}
    for r in attacked_tasks(rows):
        if r["category"] != "attacked": continue
        g = r["meta"]["goal_kind"]
        hit = any(s.get("label")=="hijacked" and addr.search(
            str((s.get("args") or {}).get("command","")) + str((s.get("args") or {}).get("path","")))
            for s in r["steps"])
        o = out.setdefault(g, [0,0]); o[0]+=1; o[1]+=int(hit)
    return out

for lab, rows in urlfree.items():
    print(f"\n{lab} (URL-free run):")
    for g, (tot, cau) in sorted(catchable_by_url_rule(rows).items()):
        print(f"  {g:18s} hijacks={tot:2d}  catchable by URL rule = {cau}/{tot}")

## 6. Figure — capability gradient

In [ ]:
import matplotlib.pyplot as plt
labs = list(data); overall = []; err = []
for lab in labs:
    n, reached, hij = rates(data[lab])
    overall.append(100*hij/n)
    lo, hi = wilson(hij, n); err.append((hi-lo)/2)
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.bar(range(len(labs)), overall, yerr=err, capsize=4,
       color=["#5B8DB8","#E08A3C","#B0413E"], width=0.55)
for i, o in enumerate(overall):
    ax.text(i, o+err[i]+0.7, f"{o:.1f}%", ha="center", fontweight="bold")
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs)
ax.set_ylabel("Hijacked (% of attacked-task episodes)")
ax.set_title("Hijack rate rises with model capability")
ax.set_ylim(0, max(overall)+max(err)+6)
plt.tight_layout(); plt.show()